# Agents — File Agent

Goal: build a minimal tool-use agent that can read and search files, then measure task completion vs. plain prompting.

In [ ]:
import sys, json, pathlib
sys.path.insert(0, '../..')

from src.client import get_client

client = get_client()
MODEL = 'claude-sonnet-4-6'

In [ ]:
TOOLS = [
    {
        "name": "read_file",
        "description": "Read the contents of a file at the given path.",
        "input_schema": {
            "type": "object",
            "properties": {"path": {"type": "string", "description": "Relative file path"}},
            "required": ["path"],
        },
    },
    {
        "name": "list_dir",
        "description": "List files in a directory.",
        "input_schema": {
            "type": "object",
            "properties": {"path": {"type": "string"}},
            "required": ["path"],
        },
    },
]


def run_tool(name: str, inputs: dict) -> str:
    p = pathlib.Path(inputs["path"])
    if name == "list_dir":
        return json.dumps([str(x) for x in p.iterdir()])
    if name == "read_file":
        return p.read_text(errors="replace")[:4000]
    return "unknown tool"


def run_agent(task: str, max_turns: int = 6) -> str:
    messages = [{"role": "user", "content": task}]
    for _ in range(max_turns):
        resp = client.messages.create(model=MODEL, max_tokens=1024, tools=TOOLS, messages=messages)
        messages.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason == "end_turn":
            return next(b.text for b in resp.content if hasattr(b, 'text'))
        tool_results = [
            {"type": "tool_result", "tool_use_id": b.id, "content": run_tool(b.name, b.input)}
            for b in resp.content if b.type == "tool_use"
        ]
        messages.append({"role": "user", "content": tool_results})
    return "max turns reached"


# example — point it at the repo root
print(run_agent("List the top-level files in ../../ and tell me what this project is about."))